In [89]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import pickle
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

In [2]:
df = pd.read_excel('data/insurance_claim2.xlsx')

In [3]:
df.head()

,Claim ID,Customer ID,Product Line,Claim Amount (NGN),Claim Date,Claim Status,Reason,Resolution Date,Insurance Company,Country,...,Days_Remaining_On_Policy,Required Documents Count,Document_Completeness_Ratio,Exclusion_Clause_Triggered,Fraud_Risk_Score,Requires_Senior_Approval,Exclusion_Risk_Index,Account Type,Account_Tier_Level,Is_Premium_Or_VIP
0,CLAIM-20250920-00001,d680466e-f4f1-4228-a842-0f5b6678afc5,Motor Vehicle,314127.66,2025-04-15,Approved,Claim verified and valid,2025-04-29,NaijaShield Assurance Plc,Nigeria,...,119,7,1.2857,0,0.0334,0,0.0225,Standard,2,0
1,CLAIM-20250920-00002,829399c2-4228-4161-89d0-18a1ba84fd81,Health Insurance,157245.23,2024-09-25,Approved,Claim verified and valid,2024-10-01,NaijaShield Assurance Plc,Nigeria,...,221,1,1.0000,0,0.1100,0,0.1900,Standard,2,0
2,CLAIM-20250920-00003,390ea33f-7ea0-41ce-a504-90bbb8f16d85,Marine & Cargo,2368230.69,2025-07-15,Approved,Claim verified and valid,2025-07-19,NaijaShield Assurance Plc,Nigeria,...,54,5,1.2000,0,0.3281,0,0.2125,Premium,3,1
3,CLAIM-20250920-00004,2c40c650-3dad-4245-baeb-097ef63c9bd3,General Accident,98568.79,2024-09-28,Approved,All policy requirements met,2024-10-03,NaijaShield Assurance Plc,Nigeria,...,282,5,1.2000,0,0.3522,0,0.0225,Standard,2,0
4,CLAIM-20250920-00005,7c77c6d2-6270-40a6-a6c5-fcfb5bef03a5,Fire & Special Perils,54964.60,2025-03-31,Approved,Claim verified and valid,2025-04-03,NaijaShield Assurance Plc,Nigeria,...,334,8,1.1250,0,0.6841,0,0.0300,Standard,2,0


In [4]:
missing = df.isnull().sum()

In [5]:
missing

Claim ID                          0
Customer ID                       0
Product Line                      0
Claim Amount (NGN)                0
Claim Date                        0
Claim Status                      0
Reason                            0
Resolution Date                1888
Insurance Company                 0
Country                           0
State                             0
City                              0
Latitude                          0
Longitude                         0
Claim Submission Channel          0
Previous Claims Count             0
Policy Tenure (Years)             0
Number of Policy Violations       0
Documents Submitted               0
Communication Preference          0
Coverage Limits (NGN)             0
Claim_to_Limit_Ratio              0
Policy Expiration Date            0
Days_past_expiration              0
Days_Past_Expiration              0
Days_Remaining_On_Policy          0
Required Documents Count          0
Document_Completeness_Ratio 

In [6]:
df.shape

(37771, 35)

In [7]:
df["Resolution Date"] = pd.to_datetime(df["Resolution Date"], errors="coerce")

In [8]:
df = df.dropna(subset=["Resolution Date"]).copy()

In [9]:
df.isnull().sum()

Claim ID                       0
Customer ID                    0
Product Line                   0
Claim Amount (NGN)             0
Claim Date                     0
Claim Status                   0
Reason                         0
Resolution Date                0
Insurance Company              0
Country                        0
State                          0
City                           0
Latitude                       0
Longitude                      0
Claim Submission Channel       0
Previous Claims Count          0
Policy Tenure (Years)          0
Number of Policy Violations    0
Documents Submitted            0
Communication Preference       0
Coverage Limits (NGN)          0
Claim_to_Limit_Ratio           0
Policy Expiration Date         0
Days_past_expiration           0
Days_Past_Expiration           0
Days_Remaining_On_Policy       0
Required Documents Count       0
Document_Completeness_Ratio    0
Exclusion_Clause_Triggered     0
Fraud_Risk_Score               0
Requires_S

In [10]:
date_columns = [
    "Claim Date",
    "Resolution Date",
    "Policy Expiration Date"
]


for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

In [11]:
numeric_columns = ["Claim Amount (NGN)",
                  "Latitude",
                  "Longitude",
                  "Previous Claims Count",
                  "Policy Tenure (Years)",
                  "Number of Policy Violations",
                  "Documents Submitted",
                  "Coverage Limits (NGN)",
                  "Claim_to_Limit_Ratio",
                  "Days_past_expiration",
                  "Days_Past_Expiration",
                  "Days_Remaining_On_Policy",
                  "Required Documents Count",
                  "Document_Completeness_Ratio",
                  "Exclusion_Clause_Triggered",
                  "Fraud_Risk_Score",
                  "Requires_Senior_Approval",
                  "Exclusion_Risk_Index",
                  "Account_Tier_Level",
                  "Is_Premium_Or_VIP"]

for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

In [12]:
df = df.drop_duplicates()

### New Fraud Risk Features

In [13]:
df["Invalid_ClaimAmount_Flag"] = (df["Claim Amount (NGN)"] < 0).fillna(False).astype(int)

In [14]:
df["Invalid_CoverageLimit_Flag"] = (df["Coverage Limits (NGN)"] <= 0).fillna(False).astype(int)

In [15]:
df["Claim_Exceeds_Limit_Flag"] = (df["Claim Amount (NGN)"] > df["Coverage Limits (NGN)"]).fillna(False).astype(int)

In [16]:
df["Invalid_Policy_Tenure_Flag"] = (df["Policy Tenure (Years)"] < 0).fillna(False).astype(int)

In [17]:
df["Invalid_Document_Count_Flag"] = ((df["Documents Submitted"] < 0) | (df["Required Documents Count"] < 0)).fillna(False).astype(int)

In [18]:
#Claim amount relative to Ratio
df["Claim_to_Coverage_ratio"] = (df["Claim Amount (NGN)"]/df["Coverage Limits (NGN)"]).replace(0, np.nan)

In [19]:
#Document Completeness calc
required = df["Required Documents Count"].replace(0, np.nan)

df["Document_Completeness_calc"] = (
    df["Documents Submitted"] / required
).clip(0, 1)

df["Document_Incomplete_Flag"] = (
    df["Documents Submitted"] < df["Required Documents Count"]
).fillna(False).astype(int)

In [20]:
# Claim datae relative to policy expiry
df["Days_from_expiry_to_claim"] = (
    df["Claim Date"] - df["Policy Expiration Date"]
).dt.days
    

### Cleaning the categorical Features

In [21]:
cat_columns = [
    "Product Line",
    "Claim Status",
    "Reason",
    "Insurance Company",
    "Country",
    "State",
    "City",
    "Claim Submission Channel",
    "Communication Preference",
    "Account Type"
]

for col in cat_columns:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip().replace("", pd.NA)

### Final Data Preprocessing for Model Development

In [22]:
#exclude identifiers and fields that may leak future decisions
exclude_columns = [
    "Claim ID",
    "Customer ID",
    "Resolution Data",
    "Resoluyion",
    "Fraud_Risk_Score",
    "Requires_Senior_Approval",
    "Claim Status",
    "Days_past_expiration",
    "Days_Past_Expiration",
    "Days_Remaining_On_Policy",
    "Claim_to_Limit_Ratio",
    "Document_Completeness_Ratio"
]

if "fraud_label" in df.columns:
    exclude_columns.append("fraud_label")

In [23]:
X = df.drop(columns=[c for c in exclude_columns if c in df.columns])

In [24]:
num_features = X.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

In [25]:
cat_features = X.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

In [26]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [27]:
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [28]:
preprocessor_pipeline = ColumnTransformer([
    ("numeric", num_pipeline, num_features),
    ("categorical", cat_pipeline, cat_features)
])

In [29]:
X_processed = preprocessor_pipeline.fit_transform(X)

In [30]:
X.shape

(35883, 34)

In [31]:
X_processed

array([[-0.39997251, -0.99688173,  1.01789617, ...,  0.        ,
         1.        ,  0.        ],
       [-0.41525992, -1.10184086,  0.32946271, ...,  0.        ,
         1.        ,  0.        ],
       [-0.19981043, -0.35065803, -0.4282014 , ...,  1.        ,
         0.        ,  0.        ],
       ...,
       [-0.42099248, -1.03193213,  0.98492151, ...,  0.        ,
         1.        ,  0.        ],
       [-0.41134429,  1.34688554,  1.30321036, ...,  0.        ,
         1.        ,  0.        ],
       [-0.4295129 ,  2.33036128,  1.12375935, ...,  0.        ,
         1.        ,  0.        ]])

In [32]:

### Determining the Fraud label field

In [45]:
conditions = [
    df["Fraud_Risk_Score"].between(0.0000, 0.3999, inclusive="both"),
    df["Fraud_Risk_Score"].between(0.4000, 1.0000, inclusive="both"),
]

categories = ["Low Risk", "High Risk"]

df["Fraud_Risk_Category"] = np.select(conditions, categories, default="Review Score")

# treating the missing fraud scores
df.loc[df["Fraud_Risk_Score"].isna(), "Fraud_Risk_Category"] = "Missing Score"

In [46]:
df.head()

,Claim ID,Customer ID,Product Line,Claim Amount (NGN),Claim Date,Claim Status,Reason,Resolution Date,Insurance Company,Country,...,Invalid_ClaimAmount_Flag,Invalid_CoverageLimit_Flag,Claim_Exceeds_Limit_Flag,Invalid_Policy_Tenure_Flag,Invalid_Document_Count_Flag,Claim_to_Coverage_ratio,Document_Completeness_calc,Document_Incomplete_Flag,Days_from_expiry_to_claim,Fraud_Risk_Category
0,CLAIM-20250920-00001,d680466e-f4f1-4228-a842-0f5b6678afc5,Motor Vehicle,314127.66,2025-04-15,Approved,Claim verified and valid,2025-04-29,NaijaShield Assurance Plc,Nigeria,...,0,0,0,0,0,0.457422,1.0,0,-119,Low Risk
1,CLAIM-20250920-00002,829399c2-4228-4161-89d0-18a1ba84fd81,Health Insurance,157245.23,2024-09-25,Approved,Claim verified and valid,2024-10-01,NaijaShield Assurance Plc,Nigeria,...,0,0,0,0,0,0.259264,1.0,0,-221,Low Risk
2,CLAIM-20250920-00003,390ea33f-7ea0-41ce-a504-90bbb8f16d85,Marine & Cargo,2368230.69,2025-07-15,Approved,Claim verified and valid,2025-07-19,NaijaShield Assurance Plc,Nigeria,...,0,0,0,0,0,0.310291,1.0,0,-54,Low Risk
3,CLAIM-20250920-00004,2c40c650-3dad-4245-baeb-097ef63c9bd3,General Accident,98568.79,2024-09-28,Approved,All policy requirements met,2024-10-03,NaijaShield Assurance Plc,Nigeria,...,0,0,0,0,0,0.352596,1.0,0,-282,Low Risk
4,CLAIM-20250920-00005,7c77c6d2-6270-40a6-a6c5-fcfb5bef03a5,Fire & Special Perils,54964.60,2025-03-31,Approved,Claim verified and valid,2025-04-03,NaijaShield Assurance Plc,Nigeria,...,0,0,0,0,0,0.644141,1.0,0,-334,High Risk


In [47]:
df['Fraud_Risk_Category'].value_counts()

Fraud_Risk_Category
Low Risk     34111
High Risk     1772
Name: count, dtype: int64

In [49]:
df1 = df.copy()

In [50]:
df['Fraud_Risk_Category'] = df['Fraud_Risk_Category'].map({
    "Low Risk": 0,
    "High Risk": 1
})

In [65]:
y = df['Fraud_Risk_Category']

In [56]:
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.30, random_state=42, stratify=y)

In [59]:
len(X_train)

25118

In [64]:
model = RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=5, class_weight="balanced", random_state=42, n_jobs=-1)

In [66]:
model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",12
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",5
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_feat

In [67]:
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

In [68]:
y_pred

array([0, 0, 0, ..., 0, 0, 0], dtype=int64)

In [69]:
y_pred_proba

array([0.46316138, 0.33974115, 0.34089323, ..., 0.20885241, 0.3962374 ,
       0.34855529])

In [70]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [71]:
accuracy = accuracy_score(y_test, y_pred)

In [72]:
accuracy

0.9522526706920575

In [76]:
importances = model.feature_importances_

In [77]:
importances

array([0.05457526, 0.05559911, 0.05517145, 0.02723574, 0.02925428,
       0.02357338, 0.03365345, 0.05831349, 0.02795315, 0.00200124,
       0.03522609, 0.00760446, 0.00293352, 0.        , 0.        ,
       0.00134305, 0.        , 0.        , 0.0637176 , 0.00420544,
       0.00107706, 0.06202178, 0.00492846, 0.00579333, 0.00384698,
       0.00342746, 0.00295063, 0.00365237, 0.00498606, 0.01330455,
       0.0017016 , 0.00110887, 0.01610947, 0.01210282, 0.23404718,
       0.00166756, 0.01544934, 0.00227144, 0.        , 0.        ,
       0.00341467, 0.00307793, 0.0024171 , 0.00304996, 0.00290756,
       0.00239139, 0.00234087, 0.00228153, 0.00259892, 0.00250257,
       0.00277518, 0.00217288, 0.00342729, 0.0033476 , 0.0026966 ,
       0.00267607, 0.00245818, 0.00255848, 0.00222444, 0.00261239,
       0.00248851, 0.00225983, 0.00281177, 0.00256105, 0.00670654,
       0.00635065, 0.00658525, 0.00698651, 0.0075763 , 0.00756273,
       0.00725285, 0.00342854, 0.00301761, 0.00504159, 0.00263

In [91]:
joblib.dump(model, 'insurance_fraud_detection_model.pkl')
#print("Model Saved")               

['insurance_fraud_detection_model.pkl']